---

<div style="padding:20px;background:linear-gradient(90deg,#12343b 0%,#2d545e 55%,#0f2027 100%);border-radius:10px;color:white;">
    <h1 style="color:white;border-bottom:none;">How to Run Module 11 Capstone Project 2</h1>
    <p style="font-size:1.2em;opacity:0.9;">Adaptive multi-hop research RAG with Groq, local embeddings, BM25, Chroma, reranking, citations, and grounding checks.</p>
</div>

---

## Project Goal

This notebook shows how to run **Module 11 - Capstone Project 2**, an adaptive multi-hop research RAG assistant.

Unlike Module 10, which focuses on a production-style RAG pipeline, this project focuses on research-style advanced RAG:

1. Plan the user question into focused retrieval sub-questions.
2. Retrieve evidence using dense vector search and BM25 keyword search.
3. Rerank retrieved evidence with a local cross-encoder.
4. Generate a citation-based answer with Groq.
5. Verify whether the answer is grounded in the retrieved context.

The stack is free-first: Groq for LLM calls, local SentenceTransformers embeddings, local Chroma vector store, BM25, and local reranking.

## Component Breakdown

| File | Purpose |
| :--- | :--- |
| `config.py` | Shared free-first settings and environment variables |
| `corpus.py` | Document loading, chunking, local embeddings, and Chroma indexing |
| `retrievers.py` | Dense + BM25 retrieval with local cross-encoder reranking |
| `graph.py` | LangGraph workflow for planning, retrieval, synthesis, and grounding verification |
| `evaluation.py` | Lightweight citation coverage and source diversity reporting |
| `demo.py` | End-to-end runnable sample project |


## Prerequisites

| Requirement | Details |
| :--- | :--- |
| Python | Python 3.13 recommended for this repository |
| Dependencies | Install with `uv sync` from the repository root |
| Groq API key | Required for planning, generation, and grounding verification |
| Embeddings | Local `sentence-transformers/all-MiniLM-L6-v2`, no paid API key needed |

Environment variables in the root `.env` file:

```ini
GROQ_API_KEY=gsk_your_key_here
GROQ_MODEL=llama-3.1-8b-instant
EMBEDDING_MODEL=sentence-transformers/all-MiniLM-L6-v2
MODULE_11_COLLECTION=module_11_research_rag
MODULE_11_CHROMA_DIR=./chroma_module_11_db
MODULE_11_RETRIEVAL_K=6
MODULE_11_RERANK_TOP_N=5
RERANKER_MODEL=cross-encoder/ms-marco-TinyBERT-L-2-v2
```

## Step 1: Install and Start Jupyter

Run these commands from the repository root in a terminal:

```bash
uv sync
uv run jupyter lab
```

Then open this notebook:

```text
Module_11_Capston_Proj_2/How_to_Run_Capstone_Proj_2.ipynb
```

In [ ]:
# Step 2: Make imports work whether Jupyter starts at the repo root or module folder.
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

module_dir = repo_root / "Module_11_Capston_Proj_2"
for path in (repo_root, module_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"Repository root: {repo_root}")
print(f"Module 11 folder: {module_dir}")

In [ ]:
# Step 3: Load environment settings.
import os
from dotenv import load_dotenv

load_dotenv(repo_root / ".env")

print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))
print("GROQ_MODEL:", os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"))
print("EMBEDDING_MODEL:", os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"))

## Step 4: Build a Small Local Knowledge Base

The demo includes a tiny built-in corpus so you can understand the workflow before indexing your own files. The first run may download local embedding and reranker model weights.

In [ ]:
from Module_11_Capston_Proj_2.demo import sample_corpus
from Module_11_Capston_Proj_2.corpus import build_vectorstore

docs = sample_corpus()
vectorstore, chunks = build_vectorstore(docs)

print(f"Raw documents: {len(docs)}")
print(f"Chunks indexed: {len(chunks)}")
print("Example chunk metadata:", chunks[0].metadata)

## Step 5: Create the Adaptive Retriever

This retriever combines:

- Dense search through Chroma and local Hugging Face embeddings
- Sparse search through BM25
- Local cross-encoder reranking for precision

In [ ]:
from Module_11_Capston_Proj_2.retrievers import AdaptiveMultiHopRetriever

retriever = AdaptiveMultiHopRetriever(vectorstore=vectorstore, corpus=chunks)

test_hits = retriever.retrieve_one("When should I use corrective RAG?")
print(f"Retrieved and reranked documents: {len(test_hits)}")
for i, doc in enumerate(test_hits[:3], 1):
    print(f"[{i}] {doc.metadata.get('source')} - {doc.page_content[:120]}...")

## Step 6: Run the LangGraph Research RAG Workflow

This step requires `GROQ_API_KEY` because Groq is used for query planning, answer generation, and grounding verification. If the key is missing, the cell will skip the LLM call gracefully.

In [ ]:
from Module_11_Capston_Proj_2.graph import build_research_graph

question = (
    "Compare corrective RAG and self-RAG for a production support assistant. "
    "Where do hybrid retrieval and reranking fit?"
)

if not os.getenv("GROQ_API_KEY"):
    print("Skipping graph run because GROQ_API_KEY is not set.")
else:
    app = build_research_graph(retriever)
    result = app.invoke({"question": question})
    print("Sub-questions:")
    for item in result.get("sub_questions", []):
        print("-", item)
    print("\nVerification:", result.get("verification"))
    print("\nAnswer:\n", result.get("answer"))

In [ ]:
# Step 7: Inspect the retrieval trace from the latest graph or retriever run.
for trace in retriever.last_trace:
    print(
        f"{trace.sub_question} | dense={trace.dense_hits}, "
        f"bm25={trace.sparse_hits}, returned={trace.returned}"
    )

In [ ]:
# Step 8: Run lightweight reporting if an answer was generated.
from Module_11_Capston_Proj_2.evaluation import print_research_rag_report

if "result" not in globals() or not result.get("answer"):
    print("No generated answer available yet. Run the graph cell after setting GROQ_API_KEY.")
else:
    print_research_rag_report(
        question=question,
        answer=result["answer"],
        docs=result.get("documents", []),
    )

## Step 9: Use Your Own Documents

To index files from the repository `data/` folder or your own document folder, use the helper below. Supported formats are `.txt`, `.md`, `.pdf`, and `.csv`.

In [ ]:
from Module_11_Capston_Proj_2.corpus import build_vectorstore_from_directory

# Uncomment to index your own documents.
# vectorstore, chunks = build_vectorstore_from_directory(str(repo_root / "data"))
# retriever = AdaptiveMultiHopRetriever(vectorstore=vectorstore, corpus=chunks)
# app = build_research_graph(retriever)
# result = app.invoke({"question": "What advanced RAG techniques are covered in these documents?"})
# print(result["answer"])

## Command-Line Alternative

You can also run the full demo from the repository root:

```bash
uv run python Module_11_Capston_Proj_2/demo.py
```

## Next Improvements

- Add metadata filters for product, document type, freshness, or owner.
- Add routing across policy docs, API docs, and support tickets.
- Persist retrieval traces for debugging and evaluation.
- Add RAGAS metrics once you have a labeled test set.
- Add a Streamlit or FastAPI interface after the core pipeline is stable.